In [1]:
import os

keys = [
    "SNOWFLAKE_ACCOUNT",
    "SNOWFLAKE_USER",
    "SNOWFLAKE_PASSWORD",
    "SNOWFLAKE_DATABASE",
    "SNOWFLAKE_SCHEMA_RAW",
    "SNOWFLAKE_SCHEMA_ANALYTICS",
    "SNOWFLAKE_WAREHOUSE",
    "SNOWFLAKE_ROLE",
]

for k in keys:
    value = os.getenv(k)
    if value:
        if "PASSWORD" in k:
            print(f"{k}: ********")
        else:
            print(f"{k}: {value}")
    else:
        print(f"{k}: MISSING")

SNOWFLAKE_ACCOUNT: ynvpgpx-snc01960
SNOWFLAKE_USER: LEO
SNOWFLAKE_PASSWORD: ********
SNOWFLAKE_DATABASE: NYC_TLC
SNOWFLAKE_SCHEMA_RAW: RAW
SNOWFLAKE_SCHEMA_ANALYTICS: ANALYTICS
SNOWFLAKE_WAREHOUSE: COMPUTE_WH
SNOWFLAKE_ROLE: SYSADMIN


In [2]:
!pip install snowflake-connector-python
!pip install -U "pyarrow>=14.0.1"

  Obtaining dependency information for snowflake-connector-python from https://files.pythonhosted.org/packages/49/8f/842946698af2903133c277611341fe23097bfd628cc3228fe16d58fc5ece/snowflake_connector_python-4.4.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 2.5 MB/s eta 0:00:00
  Obtaining dependency information for asn1crypto<2.0.0,>0.24.0 from https://files.pythonhosted.org/packages/c9/7f/09065fd9e27da0eda08b4d6897f1c13535066174cc023af248fc2a8d5e5a/asn1crypto-1.5.1-py2.py3-none-any.whl.metadata
  Obtaining dependency information for cryptography>=46.0.5 from https://files.pythonhosted.org/packages/ec/4d/8e7d7245c79c617d08724e2efa397737715ca0ec830ecb3c91e547302555/cryptography-46.0.6-cp311-abi3-manylinux_2_34_x86_64.whl.metadata
  Obtaining dependency information for pyOpenSSL>=24.0.0 from https://files.pythonhosted.org/packages/fb/7d/d4f7d908fa8415571771b30669251d57c3cf313b36a856e6d7548ae01619/pyopenssl-2

In [3]:
import os
import snowflake.connector

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    role=os.getenv("SNOWFLAKE_ROLE"),
)

cur = conn.cursor()

cur.execute(f"USE DATABASE {os.getenv('SNOWFLAKE_DATABASE')}")
cur.execute(f"USE SCHEMA {os.getenv('SNOWFLAKE_SCHEMA_RAW')}")

cur.execute("""
SELECT 
    CURRENT_VERSION(), 
    CURRENT_DATABASE(), 
    CURRENT_SCHEMA(), 
    CURRENT_WAREHOUSE(), 
    CURRENT_ROLE()
""")

print(cur.fetchone())

cur.close()
conn.close()

('10.11.1', 'NYC_TLC', 'RAW', 'COMPUTE_WH', 'SYSADMIN')


In [4]:
from pyspark.sql import SparkSession
import os

spark = (
    SparkSession.builder
    .appName("Snowflake Spark Test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "net.snowflake:spark-snowflake_2.12:2.12.0-spark_3.4",
            "net.snowflake:snowflake-jdbc:3.14.4"
        ])
    )
    .getOrCreate()
)

sfOptions = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": os.getenv("SNOWFLAKE_DATABASE"),
    "sfSchema": os.getenv("SNOWFLAKE_SCHEMA_RAW"),
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

df = spark.read \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("query", "SELECT CURRENT_VERSION() AS version") \
    .load()

df.show()

+-------+
|VERSION|
+-------+
|10.11.1|
+-------+

